### Laboratorio 2

**Grupo 38**

**Integrantes:**

Mateo Zambrano 202321531
 
Nicolas Romero 202321309

### 1. Configuración y carga de datos

#### 1.1 Importación de librerías

In [1]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, RobustScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.inspection import permutation_importance
from sklearn.preprocessing import FunctionTransformer, PolynomialFeatures
from sklearn.metrics import mean_absolute_error


warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="deep", font_scale=1.2)
plt.rcParams["figure.figsize"] = (10, 5)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", None)

#### 1.2 Constantes de reproducibilidad

In [2]:
RANDOM_STATE = 42
TEST_SIZE = 0.25
UMBRAL_MAX = 44.8
OBJETIVO = "temp_max_manana"

POLY_DEGREES = [2, 3, 4]
INCLUDE_BIAS = False

ALPHAS = np.logspace(-3, 3, 13)
L1_RATIO = 0.5

CV_FOLDS = 5
SCORING = "neg_root_mean_squared_error"

CONFIDENCE_LEVEL = 0.95
N_BOOTSTRAP = 1000

In [3]:
DATA_DIR = Path("data")
RUTA_TRAIN = DATA_DIR / "datos Lab 1.csv"
RUTA_SIN_ETIQUETA = DATA_DIR / "Datos Test Lab 1.csv"
RUTA_DICCIONARIO = DATA_DIR / "Diccionario de datos.xlsx"

datos = pd.read_csv(RUTA_TRAIN)
sin_etiqueta = pd.read_csv(RUTA_SIN_ETIQUETA)
diccionario = pd.read_excel(RUTA_DICCIONARIO)

print("Entrenamiento:", datos.shape)
print("No etiquetados: ", sin_etiqueta.shape)
print("Columnas que están en train y no en el archivo sin etiquetar:", set(datos.columns) - set(sin_etiqueta.columns))

diccionario

Entrenamiento: (2576, 27)
No etiquetados:  (364, 26)
Columnas que están en train y no en el archivo sin etiquetar: {'temp_max_manana'}


,variable,tipo,descripcion
0,fecha,texto,"Fecha de la observación, en formato día.mes.año."
1,presion_media,numérico,Presión atmosférica media del día (mbar).
2,presion_min,numérico,Presión atmosférica mínima del día (mbar).
3,presion_max,numérico,Presión atmosférica máxima del día (mbar).
4,presion_desv,numérico,Desviación típica de la presión durante el día (mbar).
5,humedad_media,numérico,Humedad relativa media del día (%).
6,humedad_min,numérico,Humedad relativa mínima del día (%).
7,humedad_max,numérico,Humedad relativa máxima del día (%).
8,humedad_desv,numérico,Desviación típica de la humedad relativa (%).
9,viento_media,numérico,Velocidad media del viento durante el día (m/s).


### 2. Exploración de datos

#### 2.1 Estructura general/exploración inicial

In [4]:
display(datos.head())
datos.info()
datos.shape

,fecha,presion_media,presion_min,presion_max,presion_desv,humedad_media,humedad_min,humedad_max,humedad_desv,viento_media,viento_min,viento_max,viento_desv,rafaga_media,rafaga_min,rafaga_max,rafaga_desv,viento_norte,viento_este,direccion_viento,registros_del_dia,anio,dia_del_anio,estacion_anio,mes,sector_viento,temp_max_manana
0,2009-01-01,999.1456,996.50,1000.87,1.3993,0.910860,0.875000,94.8,1.7650,0.7786,0.05,1.559641,0.5753,1.3783,0.25,2.255577,0.8488,-0.3618,-0.0223,183.5308,143.0,2009.0,1.0,invierno,January,S,-2.12
1,2009-01-02,999.6006,997.93,1002.65,1.5039,0.920868,86.600000,96.3,2.7588,1.4195,0.22,3.870000,0.9168,2.2274,0.63,6.130000,1.2544,0.4267,0.3689,40.8436,144.0,2009.0,2.0,invierno,JULY,NE,-0.82
2,2009-01-03,998.5486,993.05,1002.49,3.1304,76.458100,48.390000,93.9,15.1796,1.2509,0.12,3.640000,0.7299,2.0651,0.38,4.880000,0.9330,-0.6993,-0.5268,216.9916,144.0,2009.0,3.0,invierno,January,SO,-0.63
3,2009-01-04,988.5107,985.12,992.93,2.3223,89.417400,97.946704,NaN,4.4904,1.7204,0.54,2.454415,0.7023,3.5649,1.38,4.430375,1.1835,-1.1268,-1.0413,222.7419,144.0,2009.0,4.0,invierno,JANUARY,SO,-1.44
4,2009-01-05,990.4057,NaN,997.54,4.2315,86.260400,74.600000,93.2,5.3922,3.8003,1.00,7.810000,1.9521,5.9400,2.13,10.880000,NaN,2.6275,0.2874,6.2418,NaN,2009.0,5.0,East,January,N,-10.88


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2576 entries, 0 to 2575
Data columns (total 27 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   fecha              2504 non-null   object 
 1   presion_media      2501 non-null   float64
 2   presion_min        2502 non-null   float64
 3   presion_max        2512 non-null   float64
 4   presion_desv       2496 non-null   float64
 5   humedad_media      2502 non-null   float64
 6   humedad_min        2496 non-null   float64
 7   humedad_max        2490 non-null   float64
 8   humedad_desv       2515 non-null   float64
 9   viento_media       2490 non-null   float64
 10  viento_min         2485 non-null   float64
 11  viento_max         2495 non-null   float64
 12  viento_desv        2504 non-null   float64
 13  rafaga_media       2498 non-null   float64
 14  rafaga_min         2504 non-null   float64
 15  rafaga_max         2503 non-null   float64
 16  rafaga_desv        2497 

(2576, 27)

### 3. Preparación de Datos

En esta sección, se prepararán los datos de forma que los arreglos no sean dependientes del aprendizaje de otros datos, como la imputación de datos por medio de la mediana o moda.

In [5]:
copia_datos = datos.copy()

#### 3.1 Conversión de tipos y calendario derivado

In [6]:
fecha_parseada = pd.to_datetime(copia_datos["fecha"], format="%Y-%m-%d")

copia_datos["fecha"] = fecha_parseada
copia_datos["anio"] = fecha_parseada.dt.year
copia_datos["dia_del_anio"] = fecha_parseada.dt.dayofyear
copia_datos["mes"] = fecha_parseada.dt.strftime("%B").str.lower()

copia_datos["registros_del_dia"] = copia_datos["registros_del_dia"].astype("Int64")
copia_datos["anio"] = copia_datos["anio"].astype("Int64")
copia_datos["dia_del_anio"] = copia_datos["dia_del_anio"].astype("Int64")

print(copia_datos[["fecha", "anio", "mes", "dia_del_anio", "registros_del_dia"]].dtypes)

fecha                datetime64[ns]
anio                          Int64
mes                          object
dia_del_anio                  Int64
registros_del_dia             Int64
dtype: object


In [7]:
print("Fechas no parseadas:", fecha_parseada.isna().sum())

Fechas no parseadas: 72


#### 3.2 Normalización de categorías

Se estandariza a una sola forma de escritura, aquellos datos escritos de formas distintas.

In [8]:
MAPA_ESTACION = {
    "invierno": "invierno", "invernio": "invierno", "winter": "invierno",
    "primavera": "primavera", "primaveraa": "primavera", "primav": "primavera", "spring": "primavera",
    "verano": "verano", "berano": "verano", "verno": "verano", "summer": "verano",
    "otono": "otono", "otoño": "otono", "autumn": "otono", "fall": "otono",
}

copia_datos["estacion_anio"] = (
    copia_datos["estacion_anio"]
    .str.strip()
    .str.lower()
    .map(MAPA_ESTACION)
)

MAPA_SECTOR = {
    "n": "N", "norte": "N", "north": "N",
    "ne": "NE", "noreste": "NE", "northeast": "NE",
    "e": "E", "este": "E", "east": "E",
    "se": "SE", "sureste": "SE", "southeast": "SE",
    "s": "S", "sur": "S", "south": "S",
    "so": "SO", "suroeste": "SO", "southwest": "SO",
    "o": "O", "oeste": "O", "west": "O",
    "no": "NO", "noroeste": "NO", "northwest": "NO",
}

copia_datos["sector_viento"] = (
    copia_datos["sector_viento"]
    .str.strip()
    .str.lower()
    .map(MAPA_SECTOR)
)

print(copia_datos["estacion_anio"].value_counts(dropna=False))
print(copia_datos["sector_viento"].value_counts(dropna=False))

estacion_anio
primavera    566
invierno     564
otono        564
verano       552
NaN          330
Name: count, dtype: int64
sector_viento
SO     774
S      582
NE     472
O      329
N      137
E       87
SE      72
NaN     71
NO      52
Name: count, dtype: int64


#### 3.3 Correcciones de validez

Se corrigen los valores fuera de rango o en escalas distintas.

In [9]:
#Centinelas
copia_datos = copia_datos.replace([-9999, -999, 9999], np.nan)

#Presion fuera de rango
mask = (copia_datos["presion_media"] < 850) | (copia_datos["presion_media"] > 1100)
copia_datos.loc[mask, "presion_media"] = np.nan
print(f"presion_media fuera de rango: {mask.sum()} -> NaN")

#Escala mezclada en humedad (decimal -> porcentual)
for col in ["humedad_media", "humedad_min"]:
    escala_decimal = copia_datos[col] <= 1
    copia_datos.loc[escala_decimal, col] = copia_datos.loc[escala_decimal, col] * 100
    print(f"{col}: {escala_decimal.sum()} valores convertidos a escala porcentual")

#humedad_min > 100 despues de corregir
mask = copia_datos["humedad_min"] > 100
copia_datos.loc[mask, "humedad_min"] = np.nan
print(f"humedad_min > 100: {mask.sum()} -> NaN")

#rafaga_desv negativa
mask = copia_datos["rafaga_desv"] < 0
copia_datos.loc[mask, "rafaga_desv"] = np.nan
print(f"rafaga_desv negativos: {mask.sum()} -> NaN")

#rafaga_media negativa: la velocidad no puede ser negativa...
#y no es fisica como para considerar un marco de referencia :D
copia_datos.loc[copia_datos["rafaga_media"] < 0, "rafaga_media"] = np.nan

#rafaga_desv raramente alta
copia_datos.loc[copia_datos["rafaga_desv"] > 20, "rafaga_desv"] = np.nan

#viento_desv: error de escala de un factor de 100
mask = copia_datos["viento_desv"] > 20
copia_datos.loc[mask, "viento_desv"] = copia_datos.loc[mask, "viento_desv"] / 100

#minimos que superan a los maximos
copia_datos.loc[copia_datos["viento_min"] > copia_datos["viento_max"], "viento_min"] = np.nan
copia_datos.loc[copia_datos["rafaga_min"] > copia_datos["rafaga_max"], "rafaga_min"] = np.nan


presion_media fuera de rango: 6 -> NaN
humedad_media: 776 valores convertidos a escala porcentual
humedad_min: 621 valores convertidos a escala porcentual
humedad_min > 100: 23 -> NaN
rafaga_desv negativos: 258 -> NaN


#### 3.4 Eliminacion de filas

Se eliminan las filas que contienen datos corruptos (invalidos codependientes) o están repetidos.

In [10]:
print("Filas al inicio: ", len(copia_datos))

#Duplicados exactos
copia_datos = copia_datos.drop_duplicates()
print("Sin duplicados exactos:", len(copia_datos))

#registros_del_dia invalidos
copia_datos = copia_datos[~(copia_datos["registros_del_dia"] > 144).fillna(False)]
print("Sin registros_del_dia > 144:", len(copia_datos))

#fechas repetidas: conservar la de mas registros
copia_datos = copia_datos.sort_values("registros_del_dia", ascending=False)
copia_datos = copia_datos.drop_duplicates(subset="fecha", keep = "first")
print("Sin fechas repetidas:", len(copia_datos))

#temperaturas fisicamente imposibles
mask = copia_datos[OBJETIVO] > UMBRAL_MAX
copia_datos = copia_datos[~mask]
print(f"Sin temperaturas > {UMBRAL_MAX}:", len(copia_datos))

#Filas sin objetivo
copia_datos = copia_datos.dropna(subset=[OBJETIVO])
print("Sin filas sin objetivo:", len(copia_datos))

#Reordenar cronologicamente
copia_datos = copia_datos.sort_values("fecha").reset_index(drop=True)
print("\nFilas finales:", len(copia_datos))


Filas al inicio:  2576
Sin duplicados exactos: 2572
Sin registros_del_dia > 144: 2568
Sin fechas repetidas: 2483
Sin temperaturas > 44.8: 2479
Sin filas sin objetivo: 2386

Filas finales: 2386


#### 3.5 Estado de valores faltantes

Se verifican que valores aún hacen falta y que serán imputados dentro de la pipeline.


In [11]:
faltantes = copia_datos.isnull().sum().sort_values(ascending=False)
cols_relevantes = faltantes.index.difference(["fecha", "anio", "registros_del_dia", OBJETIVO])
print(faltantes[cols_relevantes][faltantes[cols_relevantes] > 0])
print("\nEstos serán imputados dentro del pipeline con datos de entrenamiento")

dia_del_anio          1
direccion_viento     73
estacion_anio       302
humedad_desv         58
humedad_max          79
humedad_media        72
humedad_min          92
mes                   1
presion_desv         78
presion_max          59
presion_media        73
presion_min          67
rafaga_desv         306
rafaga_max           68
rafaga_media         74
rafaga_min          182
sector_viento        64
viento_desv          66
viento_este          57
viento_max           75
viento_media         81
viento_min          196
viento_norte         72
dtype: int64

Estos serán imputados dentro del pipeline con datos de entrenamiento


#### 3.6 Separación de variables

##### 3.6.1 Columnas eliminadas

In [13]:
COLUMNAS_ELIMINADAS = ["fecha"]

X = copia_datos.drop(columns=[OBJETIVO] + COLUMNAS_ELIMINADAS)
y = copia_datos[OBJETIVO]

X

,presion_media,presion_min,presion_max,presion_desv,humedad_media,humedad_min,humedad_max,humedad_desv,viento_media,viento_min,viento_max,viento_desv,rafaga_media,rafaga_min,rafaga_max,rafaga_desv,viento_norte,viento_este,direccion_viento,registros_del_dia,anio,dia_del_anio,estacion_anio,mes,sector_viento
0,999.1456,996.50,1000.87,1.3993,91.0860,87.500000,94.8,1.7650,0.77860,0.050,1.559641,0.5753,1.3783,0.25,2.255577,0.8488,-0.3618,-0.0223,183.5308,143,2009,1,invierno,january,S
1,999.6006,997.93,1002.65,1.5039,92.0868,86.600000,96.3,2.7588,1.41950,0.220,3.870000,0.9168,2.2274,0.63,6.130000,1.2544,0.4267,0.3689,40.8436,144,2009,2,invierno,january,NE
2,998.5486,993.05,1002.49,3.1304,76.4581,48.390000,93.9,15.1796,1.25090,0.120,3.640000,0.7299,2.0651,0.38,4.880000,0.9330,-0.6993,-0.5268,216.9916,144,2009,3,invierno,january,SO
3,988.5107,985.12,992.93,2.3223,89.4174,97.946704,NaN,4.4904,1.72040,0.540,2.454415,0.7023,3.5649,1.38,4.430375,1.1835,-1.1268,-1.0413,222.7419,144,2009,4,invierno,january,SO
4,990.4057,NaN,997.54,4.2315,86.2604,74.600000,93.2,5.3922,3.80030,1.000,7.810000,1.9521,5.9400,2.13,10.880000,NaN,2.6275,0.2874,6.2418,<NA>,2009,5,NaN,january,N
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2381,1003.5725,1002.35,1004.96,0.7294,87.1097,72.100000,97.7,7.5170,3.02724,0.792,9.504000,0.4208,1.4214,0.48,3.840000,NaN,-0.6644,-0.0853,187.3193,144,2015,362,verano,december,S
2382,1002.1544,1000.83,1005.33,1.2520,95.0910,79.400000,99.5,5.3604,0.86610,0.190,2.340000,0.5179,NaN,0.44,3.560000,0.7173,-0.5544,-0.0871,188.9267,144,2015,363,invierno,december,S
2383,1002.9883,997.51,1005.73,2.5524,84.4737,NaN,99.4,12.9567,2.12740,0.310,5.160000,1.2963,3.4318,NaN,0.640000,2.0483,-2.0540,-0.1130,183.1497,144,2015,364,invierno,december,S
2384,996.9224,995.01,999.20,1.4003,72.6064,49.560000,97.4,17.6803,3.11310,0.740,5.700000,1.3775,NaN,NaN,9.090000,1.9769,-3.0401,0.1673,176.8493,144,2015,365,invierno,december,S


In [15]:
y

0       -2.12
1       -0.82
2       -0.63
3       -1.44
4      -10.88
        ...  
2381     9.16
2382     5.65
2383     2.92
2384     2.72
2385    11.05
Name: temp_max_manana, Length: 2386, dtype: float64

### 3.7 Columnas numéricas y categóricas

Se identifican las columnas numéricas y categóricas, ya que el tipo de variable determina los pasos de transformación y se aplicarán a cada una dentro del pipeline.

In [16]:
columnas_numericas = [
    "presion_media", "presion_min", "presion_max", "presion_desv",
    "humedad_media", "humedad_min", "humedad_max", "humedad_desv",
    "viento_media", "viento_min", "viento_max", "viento_desv",
    "rafaga_media", "rafaga_min", "rafaga_max", "rafaga_desv",
    "viento_norte", "viento_este", "direccion_viento",
    "dia_del_anio",
]

columnas_categoricas = [
    "estacion_anio", "mes", "sector_viento",
]

print("Numéricas:", len(columnas_numericas))
print("Categóricas:", len(columnas_categoricas))
print("Total:", len(columnas_numericas) + len(columnas_categoricas), "de", X.shape[1])

Numéricas: 20
Categóricas: 3
Total: 23 de 25
